# Lab 06 — تحسين التكلفة وزمن الاستجابة

**نظرة عامة:** تقليل تكلفة النموذج وزمن الاستجابة باستخدام Caching وSmart Routing، مع تخصيص النموذج الأقوى للطلبات المعقدة أو الطارئة.

راجع ملف `README.md` داخل هذا المجلد للحصول على شرح مختصر ومنظم للأهداف، طريقة العمل، النتائج، وأهم النقاط التعليمية.


In [1]:
ASSISTANT_NAME = "Farah Hospital AI"

PRICE_PER_TOKEN = {
    "flagship": 10,
    "cheap": 2
}

LATENCY = {
    "flagship": 1.2,
    "cheap": 0.4
}

COMPLEX_WORDS = [
    "emergency",
    "critical",
    "severe",
    "urgent",
    "chest pain"
]

DEMO_ANSWER = (
    "Your hospital request has been received "
    "and will be processed safely."
)

print("🏥 Assistant :", ASSISTANT_NAME)
print("💰 Prices    :", PRICE_PER_TOKEN)
print("⏱️ Latency   :", LATENCY)
print("🧠 Complex words:", COMPLEX_WORDS)
print("✅ Optimisation system ready")

🏥 Assistant : Farah Hospital AI
💰 Prices    : {'flagship': 10, 'cheap': 2}
⏱️ Latency   : {'flagship': 1.2, 'cheap': 0.4}
🧠 Complex words: ['emergency', 'critical', 'severe', 'urgent', 'chest pain']
✅ Optimisation system ready


In [2]:
REPLAY = [
    "How can I book a doctor appointment?",
    "What are the hospital working hours?",
    "How can I book a doctor appointment?",
    "I have severe chest pain and need urgent help",
    "What are the hospital working hours?",
    "How can I check my patient record?",
    "How can I book a doctor appointment?",
    "I have a critical emergency",
    "How can I check my patient record?"
]

print("📩 Total messages :", len(REPLAY))
print("🔁 Unique messages:", len(set(REPLAY)))
print("♻️ Repeated messages:", len(REPLAY) - len(set(REPLAY)))

📩 Total messages : 9
🔁 Unique messages: 5
♻️ Repeated messages: 4


In [3]:
def count_tokens(text):
    return len(text.split())


def call_model(model, message):

    input_tokens = count_tokens(message)
    output_tokens = count_tokens(DEMO_ANSWER)

    tokens = input_tokens + output_tokens

    cost = tokens * PRICE_PER_TOKEN[model]

    seconds = LATENCY[model]

    return DEMO_ANSWER, cost, seconds


print(
    call_model(
        "flagship",
        "How can I book a doctor appointment?"
    )
)

print(
    call_model(
        "cheap",
        "How can I book a doctor appointment?"
    )
)

('Your hospital request has been received and will be processed safely.', 180, 1.2)
('Your hospital request has been received and will be processed safely.', 36, 0.4)


In [4]:
def choose_model(message):

    message_lower = message.lower()

    for word in COMPLEX_WORDS:

        if word in message_lower:

            print(
                "🚨 Complex request detected → FLAGSHIP"
            )

            return "flagship"

    print(
        "⚡ Simple request detected → CHEAP"
    )

    return "cheap"


print(
    choose_model(
        "How can I book a doctor appointment?"
    )
)

print(
    choose_model(
        "I have severe chest pain and need urgent help"
    )
)

⚡ Simple request detected → CHEAP
cheap
🚨 Complex request detected → FLAGSHIP
flagship


In [5]:
CACHE = {}


def get_cached_answer(message):

    if message in CACHE:
        return CACHE[message]

    return None


def save_to_cache(message, answer):

    CACHE[message] = answer


print("♻️ Cache system ready")

♻️ Cache system ready


In [6]:
def run_replay(use_cache, use_router):

    cache = {}

    total_cost = 0
    total_seconds = 0

    model_calls = 0
    cache_hits = 0

    flagship_calls = 0
    cheap_calls = 0


    for message in REPLAY:

        # CACHE CHECK
        if use_cache and message in cache:

            cache_hits += 1

            print(
                "♻️ CACHE HIT |",
                message
            )

            continue


        # DEFAULT MODEL
        model = "flagship"


        # SMART ROUTER
        if use_router:
            model = choose_model(message)


        # MODEL CALL
        answer, cost, seconds = call_model(
            model,
            message
        )


        model_calls += 1

        total_cost += cost
        total_seconds += seconds


        if model == "flagship":
            flagship_calls += 1

        else:
            cheap_calls += 1


        # SAVE RESULT
        cache[message] = answer


        print(
            "🤖 MODEL CALL |",
            model.upper(),
            "|",
            message
        )


    return {
        "cost": total_cost,
        "seconds": round(total_seconds, 1),
        "model_calls": model_calls,
        "cache_hits": cache_hits,
        "flagship_calls": flagship_calls,
        "cheap_calls": cheap_calls
    }

In [7]:
print("================================")
print("🔴 BASELINE SYSTEM")
print("No Cache + No Router")
print("================================")

before = run_replay(
    use_cache=False,
    use_router=False
)

print()
print("BASELINE RESULTS")
print(before)

🔴 BASELINE SYSTEM
No Cache + No Router
🤖 MODEL CALL | FLAGSHIP | How can I book a doctor appointment?
🤖 MODEL CALL | FLAGSHIP | What are the hospital working hours?
🤖 MODEL CALL | FLAGSHIP | How can I book a doctor appointment?
🤖 MODEL CALL | FLAGSHIP | I have severe chest pain and need urgent help
🤖 MODEL CALL | FLAGSHIP | What are the hospital working hours?
🤖 MODEL CALL | FLAGSHIP | How can I check my patient record?
🤖 MODEL CALL | FLAGSHIP | How can I book a doctor appointment?
🤖 MODEL CALL | FLAGSHIP | I have a critical emergency
🤖 MODEL CALL | FLAGSHIP | How can I check my patient record?

BASELINE RESULTS
{'cost': 1600, 'seconds': 10.8, 'model_calls': 9, 'cache_hits': 0, 'flagship_calls': 9, 'cheap_calls': 0}


In [8]:
print("================================")
print("🟡 OPTIMISATION 1: CACHE")
print("Cache ON + Router OFF")
print("================================")

cache_only = run_replay(
    use_cache=True,
    use_router=False
)

print()
print("CACHE RESULTS")
print(cache_only)

🟡 OPTIMISATION 1: CACHE
Cache ON + Router OFF
🤖 MODEL CALL | FLAGSHIP | How can I book a doctor appointment?
🤖 MODEL CALL | FLAGSHIP | What are the hospital working hours?
♻️ CACHE HIT | How can I book a doctor appointment?
🤖 MODEL CALL | FLAGSHIP | I have severe chest pain and need urgent help
♻️ CACHE HIT | What are the hospital working hours?
🤖 MODEL CALL | FLAGSHIP | How can I check my patient record?
♻️ CACHE HIT | How can I book a doctor appointment?
🤖 MODEL CALL | FLAGSHIP | I have a critical emergency
♻️ CACHE HIT | How can I check my patient record?

CACHE RESULTS
{'cost': 890, 'seconds': 6.0, 'model_calls': 5, 'cache_hits': 4, 'flagship_calls': 5, 'cheap_calls': 0}


In [9]:
print("================================")
print("🟠 OPTIMISATION 2: ROUTER")
print("Cache OFF + Router ON")
print("================================")

router_only = run_replay(
    use_cache=False,
    use_router=True
)

print()
print("ROUTER RESULTS")
print(router_only)

🟠 OPTIMISATION 2: ROUTER
Cache OFF + Router ON
⚡ Simple request detected → CHEAP
🤖 MODEL CALL | CHEAP | How can I book a doctor appointment?
⚡ Simple request detected → CHEAP
🤖 MODEL CALL | CHEAP | What are the hospital working hours?
⚡ Simple request detected → CHEAP
🤖 MODEL CALL | CHEAP | How can I book a doctor appointment?
🚨 Complex request detected → FLAGSHIP
🤖 MODEL CALL | FLAGSHIP | I have severe chest pain and need urgent help
⚡ Simple request detected → CHEAP
🤖 MODEL CALL | CHEAP | What are the hospital working hours?
⚡ Simple request detected → CHEAP
🤖 MODEL CALL | CHEAP | How can I check my patient record?
⚡ Simple request detected → CHEAP
🤖 MODEL CALL | CHEAP | How can I book a doctor appointment?
🚨 Complex request detected → FLAGSHIP
🤖 MODEL CALL | FLAGSHIP | I have a critical emergency
⚡ Simple request detected → CHEAP
🤖 MODEL CALL | CHEAP | How can I check my patient record?

ROUTER RESULTS
{'cost': 608, 'seconds': 5.2, 'model_calls': 9, 'cache_hits': 0, 'flagship_calls'

In [10]:
print("================================")
print("🟢 FULL OPTIMISATION")
print("Cache ON + Smart Router ON")
print("================================")

after = run_replay(
    use_cache=True,
    use_router=True
)

print()
print("OPTIMISED RESULTS")
print(after)

🟢 FULL OPTIMISATION
Cache ON + Smart Router ON
⚡ Simple request detected → CHEAP
🤖 MODEL CALL | CHEAP | How can I book a doctor appointment?
⚡ Simple request detected → CHEAP
🤖 MODEL CALL | CHEAP | What are the hospital working hours?
♻️ CACHE HIT | How can I book a doctor appointment?
🚨 Complex request detected → FLAGSHIP
🤖 MODEL CALL | FLAGSHIP | I have severe chest pain and need urgent help
♻️ CACHE HIT | What are the hospital working hours?
⚡ Simple request detected → CHEAP
🤖 MODEL CALL | CHEAP | How can I check my patient record?
♻️ CACHE HIT | How can I book a doctor appointment?
🚨 Complex request detected → FLAGSHIP
🤖 MODEL CALL | FLAGSHIP | I have a critical emergency
♻️ CACHE HIT | How can I check my patient record?

OPTIMISED RESULTS
{'cost': 466, 'seconds': 3.6, 'model_calls': 5, 'cache_hits': 4, 'flagship_calls': 2, 'cheap_calls': 3}


In [11]:
print("""
========================================
      🏥 FARAH HOSPITAL AI
       OPTIMISATION REPORT
========================================
""")

print("BEFORE OPTIMISATION")
print("-------------------")
print("Cost       :", before["cost"])
print("Time       :", before["seconds"], "seconds")
print("Model calls:", before["model_calls"])

print()

print("AFTER OPTIMISATION")
print("------------------")
print("Cost       :", after["cost"])
print("Time       :", after["seconds"], "seconds")
print("Model calls:", after["model_calls"])
print("Cache hits :", after["cache_hits"])
print("Flagship   :", after["flagship_calls"])
print("Cheap      :", after["cheap_calls"])


      🏥 FARAH HOSPITAL AI
       OPTIMISATION REPORT

BEFORE OPTIMISATION
-------------------
Cost       : 1600
Time       : 10.8 seconds
Model calls: 9

AFTER OPTIMISATION
------------------
Cost       : 466
Time       : 3.6 seconds
Model calls: 5
Cache hits : 4
Flagship   : 2
Cheap      : 3


In [12]:
cost_saved = before["cost"] - after["cost"]

cost_reduction = (
    cost_saved
    / before["cost"]
    * 100
)


time_saved = (
    before["seconds"]
    - after["seconds"]
)

time_reduction = (
    time_saved
    / before["seconds"]
    * 100
)


print("💰 Cost saved:", cost_saved)

print(
    "📉 Cost reduction:",
    round(cost_reduction, 1),
    "%"
)

print(
    "⏱️ Time saved:",
    round(time_saved, 1),
    "seconds"
)

print(
    "🚀 Latency reduction:",
    round(time_reduction, 1),
    "%"
)

💰 Cost saved: 1134
📉 Cost reduction: 70.9 %
⏱️ Time saved: 7.2 seconds
🚀 Latency reduction: 66.7 %


In [13]:
cost_saved = before["cost"] - after["cost"]

cost_reduction = (
    cost_saved
    / before["cost"]
    * 100
)


time_saved = (
    before["seconds"]
    - after["seconds"]
)

time_reduction = (
    time_saved
    / before["seconds"]
    * 100
)


print("💰 Cost saved:", cost_saved)

print(
    "📉 Cost reduction:",
    round(cost_reduction, 1),
    "%"
)

print(
    "⏱️ Time saved:",
    round(time_saved, 1),
    "seconds"
)

print(
    "🚀 Latency reduction:",
    round(time_reduction, 1),
    "%"
)

💰 Cost saved: 1134
📉 Cost reduction: 70.9 %
⏱️ Time saved: 7.2 seconds
🚀 Latency reduction: 66.7 %


In [14]:
print("""
╔════════════════════════════════════════════╗
║          FARAH HOSPITAL AI 🏥              ║
║        OPTIMISATION DASHBOARD              ║
╠════════════════════════════════════════════╣
""")

print(
    "║ Baseline Cost      :", before["cost"]
)

print(
    "║ Optimised Cost     :", after["cost"]
)

print(
    "║ Cost Reduction     :",
    round(cost_reduction, 1),
    "%"
)

print(
    "║ Baseline Time      :",
    before["seconds"],
    "sec"
)

print(
    "║ Optimised Time     :",
    after["seconds"],
    "sec"
)

print(
    "║ Latency Reduction  :",
    round(time_reduction, 1),
    "%"
)

print(
    "║ Cache Hits         :",
    after["cache_hits"]
)

print(
    "║ Flagship Calls     :",
    after["flagship_calls"]
)

print(
    "║ Cheap Model Calls  :",
    after["cheap_calls"]
)

print("""
╠════════════════════════════════════════════╣
║ STATUS: OPTIMISED ✅                       ║
╚════════════════════════════════════════════╝
""")


╔════════════════════════════════════════════╗
║          FARAH HOSPITAL AI 🏥              ║
║        OPTIMISATION DASHBOARD              ║
╠════════════════════════════════════════════╣

║ Baseline Cost      : 1600
║ Optimised Cost     : 466
║ Cost Reduction     : 70.9 %
║ Baseline Time      : 10.8 sec
║ Optimised Time     : 3.6 sec
║ Latency Reduction  : 66.7 %
║ Cache Hits         : 4
║ Flagship Calls     : 2
║ Cheap Model Calls  : 3

╠════════════════════════════════════════════╣
║ STATUS: OPTIMISED ✅                       ║
╚════════════════════════════════════════════╝

